<a href="https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii"
REPO_DIR = "flyrank-ml-internship-hadii"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

Loaded: (30000, 44)


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a ranking/scoring task. My lane needs an ordered list of pages so an editor can work through them in priority order, not a yes/no label on each page.

In [ ]:
# Sketch of the output shape for a ranking/scoring task
import pandas as pd
example = pd.DataFrame({
    "content_id": ["page_A", "page_B", "page_C"],
    "priority_score": [92, 71, 15]
}).sort_values("priority_score", ascending=False)
example

,content_id,priority_score
0,page_A,92
1,page_B,71
2,page_C,15


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

This is a ranking/scoring task, not classification. My lane's question is "which pages should be reviewed first," so the output an editor needs is an ordered list with a priority number per page, not a single yes/no label. My target is a proxy "refresh priority" score built from observed signals (trend_direction, days_since_last_update, search_volume), not a directly observed label, since nobody manually tagged "needs refresh."

In [ ]:
cols = ["content_id", "trend_direction", "trend_pct", "days_since_last_update",
        "word_count", "search_volume", "impressions_90d"]
df[cols].head(10)


,content_id,trend_direction,trend_pct,days_since_last_update,word_count,search_volume,impressions_90d
0,content_304f48230142,down,-41.4,20,3221.0,10.0,3803
1,content_a1fb4e703a9e,down,-57.7,25,2481.0,90.0,15320
2,content_9aa793d4d895,down,-60.9,20,3515.0,0.0,12581
3,content_331d6c4de07b,stable,-13.8,22,NaN,10.0,11751
4,content_d99b7a2d90ca,down,-34.7,14,2803.0,0.0,19140
5,content_d4084a4bc775,down,-38.9,20,3080.0,720.0,3970
6,content_9a34b442b552,down,-92.3,20,3059.0,0.0,20
7,content_a63219c6e95a,stable,0.6,22,NaN,590.0,1724
8,content_5e6c160719bc,down,-58.8,20,3807.0,0.0,32574
9,content_c27558df2b0c,down,-29.2,104,NaN,0.0,1240


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My success metric is precision@50: of the top 50 pages my score ranks first, what fraction are actually declining or losing demand. This matters because an editor only has time to review a fixed batch, so what counts is whether the top of the list is right, not the entire ranking.

In [ ]:
print("Share of pages currently declining:", (df["trend_direction"] == "down").mean())


Share of pages currently declining: 0.5420666666666667


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page, identified by content_id. Each page has its own traffic history, freshness, and demand numbers, which is the level I'm scoring and ranking at.

In [ ]:
df[["content_id", "client_id", "trend_direction", "trend_pct",
    "days_since_last_update", "word_count", "search_volume"]].sample(5, random_state=1)


,content_id,client_id,trend_direction,trend_pct,days_since_last_update,word_count,search_volume
10747,content_27441bff4885,client_349c41201b,down,-41.4,20,2856.0,30.0
12573,content_6352ef618f56,client_4e07408562,down,-90.7,20,2906.0,20.0
29676,content_5f43c51e146a,client_3fdba35f04,down,-100.0,104,1451.0,70.0
8856,content_4e64474fabfa,client_3fdba35f04,down,-44.7,104,1393.0,10.0
21098,content_d0b2b5a0ef58,client_19581e27de,up,89.2,20,3095.0,0.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "declining and stale" would flag a huge, mixed bag of pages, from ones with real demand to ones almost nobody searches for. Trend, freshness, demand, and content length interact differently across content types, so a single hardcoded threshold either misses real opportunities or drowns the list in noise. A model can weigh and combine these signals instead of guessing one cutoff.

In [ ]:
rule_flag = (df["trend_direction"] == "down") & (df["days_since_last_update"] > 90)
print("Rule flags:", rule_flag.sum(), "pages")
df.loc[rule_flag, "search_volume"].describe()


Rule flags: 5686 pages


,search_volume
count,5487.000000
mean,112.513213
std,1407.932182
min,0.000000
25%,0.000000
50%,10.000000
75%,20.000000
max,60500.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.